# Dating Apps Reviews EDA and Sentiment Analysis

## Introduction
Social media was invented in 1997 to connect people, so people can find someone to talk with, someone to play a game together, or someone to become more than just a friend.

<blockquote>"When Tinder became available to all smartphone users in 2013, it ushered in a new era in the history of romance.<br>
- Ashley Fetters, https://www.theatlantic.com/family/archive/2018/12/tinder-changed-dating/578698/</blockquote>

In this digital era, dating apps are something common. Some of the reasons most people use dating apps are looking for love, sex, or casual dating. Dating apps became more popular after Tinder expanded to Android phones in 2013 and shortly thereafter, many more dating apps came online.

Dating app users are varied. From high school students to 60 years old, even though most app has limited their user age minimum at 18 years old. The number of people using dating apps is increasing every year, especially when Covid-19 came in 2020 and forced the whole world into lockdown. According to statista.com,

<blockquote>"Tinder is the most downloaded dating application on Android devices worldwide. As of October 2020, the app was downloaded nearly 2.6 million times. Badoo ranked second with roughly 1.8 million downloads in the evaluated period, followed by Bumble and happn."</blockquote>

The way dating apps came and changed the shape of dating forever made me have several questions. I'm trying to answer them via this simple project. Let's get to it.

## Questions
1. From 2013 to 2022, according to the dataset, in which month and year are most dating apps used??
2. From ratings 1 to 5, which rating has the most negative reviews and why?

## Import Libraries

In [ ]:
import pandas
import numpy
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs

In [ ]:
data = pandas.read_csv('/kaggle/input/datingappreviews/DatingAppReviewsDataset.csv')

In [ ]:
data.head()

## EDA And Data Cleaning

First we going to do an EDA to understand the dataset, and a data cleaning. We can start EDA by calling the <code>.shape</code> function to know the total of rows and columns.

In [ ]:
data.shape

And then use the <code>.info()</code> to look at the data types and non-null values.

In [ ]:
data.info()

In [ ]:
data.isna().sum()

As you can see, we have some null values. We will clean it later because it doesn't affect the EDA right now.

In [ ]:
data.describe()

In [ ]:
data.tail()

Change the <b>['Unnamed: 0']</b> to <b>id</b>.

In [ ]:
data.rename(columns={'Unnamed: 0':'id'},inplace=True)

To find out in what year the apps were used the most, we can use a line plot. But before we create a line plot, i want to seperate the time and date.

In [ ]:
data["Date"] = pandas.to_datetime(data['Date&Time']).dt.date

In [ ]:
data["Time"] = pandas.to_datetime(data['Date&Time']).dt.time

In [ ]:
data = data.drop("Date&Time",axis=1)

In [ ]:
data.head()

In [ ]:
data.info()

After that we can do groupby by calling the <code>.groupby()</code> function to group the data by month and then use <code>.count()</code> to count the sum of data in each month.

# Answers
## 1. From 2013 to 2022, according to the dataset, in which month and year are most dating apps used?
To answer this question, I'm going to use a line plot. Because other than I want to know the answer, I also want to see the growth of the dating apps' total usage per month from 2013 to 2022. So the first thing we have to do to create a line plot is group the data based on its year and assign it to a variable called <b>quick_recap</b>. Also use the <code>.count()</code> function to get the total values of each month.

In [ ]:
quick_recap=data.groupby(['Date'])['Date'].count()

Let's call it to see what the data looks like.

In [ ]:
quick_recap

Plot a line plot to show the growth.

In [ ]:
fig = px.line(quick_recap,title="Growth of the dating apps' total usage per month from 2013 to 2022",template="plotly_dark")
fig.update_traces(line_color='red')
fig.show()

From the line plot, we know that the highest value is in November 2016, when it had around 1672 users. Meanwhile, the lowest value was in May 2013, with only five users. We can also create a pie plot to see the distribution of the apps in the data, so we can know which app was most used.

In [ ]:
fig = px.pie(data,values=data['App'].value_counts().values,
             names=data['App'].value_counts().index,
             title='Percentage Of Dating Apps Used Between February 2013 To December 2022')
fig.show()

From the pie plot above, we know that 77% of the users use Tinder.

## 2. From ratings 1 to 5, which rating has the most negative reviews and why?

To answer this, we are going to do a quick and simple sentiment analysis using transformers. First, we have to import the libraries that we need.

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

Remember that based on the null-values check that we ran before, we have some null values. There are some methods that we can use to remove null values from a dataframe, in this project we are going to remove the null values by drop them using the <code>.dropna()</code> function.

In [ ]:
data.isna().sum()

In [ ]:
data.dropna(inplace=True)

After we called the <code>.dropna()</code> function, we can see that there are no more null values in our data.

In [ ]:
data.isna().sum()

First, we are going to train the model using the Twitter-Roberta-base-sentiment from huggingface official website. All we have to do is specified the dataset and assign it to a variable, and then we can use it on the model. We also have to create a tokenizer to splitting the raw text into small chunks of words or sentences, called tokens.

In [ ]:
model_from_dataset = f"cardiffnlp/twitter-roberta-base-sentiment"
token = AutoTokenizer.from_pretrained(model_from_dataset)
model = AutoModelForSequenceClassification.from_pretrained(model_from_dataset)

After that, we'd make a function that is able to detect the sentiment of the data that we pass via the parameters. So how this function work is first it will get the text from the parameter, and then split the raw text into small chunks, pass it to the model, and it'd return values of the negative, neutral, and positive. If, for example, the text is negative then the negative value would be closer to one, and the same goes on for the other sentiment. Meanwhile, if the text isn't negative, then the value would be closer to zero.

In [ ]:
def roberta(text):
    encoded_text = token(text, return_tensors='pt')
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
    "Negative":scores[0],
    "Neutral":scores[1],
    "Positive":scores[2]
    }
    return scores_dict

Testing the model we've just made using an example.

In [ ]:
example = data['Review'][0]
example

Calling the function and pass the parameter.

In [ ]:
roberta(example)

From our point of view, just by reading the sentence, we'd know that the example text is a negative sentence. But the computer doesn't know. That's why it must learn first before deciding whether the sentence is negative, neutral, or positive. We can see that the model returns a couple of values. And the negative value has the highest score. It means the computer also detected the text as a negative sentence.

Because the data is so big, let's create a subset of the data consist of 2000 rows to analyze.

In [ ]:
data_subset=data.head(2000)

In [ ]:
data_subset.head()

In [ ]:
data_subset.shape

Looping to do a calculation of each text/value.

In [ ]:
roberta_result={}
for i in range(1,len(data_subset)):
    roberta_result[i]=0
    i+=1
a=1
try:
    for text in data_subset['Review']: 
        res = roberta(text)
        roberta_result[a]=res
        a+=1
except RuntimeError:
    print('Number : {}'.format(a))
print(roberta_result)

Merge the result.

In [ ]:
data_subset_roberta_result = pandas.DataFrame(roberta_result).T
data_subset_roberta_result = data_subset_roberta_result.reset_index().rename(columns={'index':'id'})
data_subset_roberta_result = data_subset_roberta_result.merge(data_subset, how='left')

In [ ]:
data_subset.shape

In [ ]:
data_subset_roberta_result.head()

Make a "Pair Plot" using a scatterplot in plotly.

In [ ]:
fig = make_subplots(rows=1, cols=3)

fig.add_trace(
    plotly.graph_objs.Scatter(x=data_subset_roberta_result["Rating"],
               y=data_subset_roberta_result["Negative"],              
               mode="markers",
               name="Negatives"),
    row=1, col=1
)

fig.add_trace(
    plotly.graph_objs.Scatter(x=data_subset_roberta_result["Rating"],
               y=data_subset_roberta_result["Neutral"],  
               mode="markers",
               name="Neutrals"),
    row=1, col=2
)

fig.add_trace(
    plotly.graph_objs.Scatter(x=data_subset_roberta_result["Rating"], 
               y=data_subset_roberta_result["Positive"], 
               mode="markers", 
               name="Positives",),
    row=1, col=3
)


fig.show()

From the plot we can see that rating 1 has the most negative reviews. Let's see some of the reviews.

In [ ]:
one = data_subset_roberta_result[(data_subset_roberta_result["Negative"]>=0.7) &
                            (data_subset_roberta_result["Rating"]==1)].tail()

In [ ]:
one

In [ ]:
one.shape

In [ ]:
one['Review'][1984]

In [ ]:
one['Review'][1985]

So the negative reviews caused by the bugs in the app. And from the reviews we got, all users are use Tinder. Other than that, we can see a lot of negative reviews in rating 5. Let's see some of them.

In [ ]:
data_subset_roberta_result[(data_subset_roberta_result["Negative"]>=0.8) &
                           (data_subset_roberta_result["Rating"]==5)]

Some of the negative reviews in rating 5 are because of wrong calculations. We can improve this by making a fine-tuning model or doing cross-validation.

# Conclusion
1. November 2016 had the most users, and May 2013 had the least users.
2. 70% of the users use Tinder.
3. Most of the negative reviews in rating 1, were caused by the bugs in the app.
